In [1]:
"""
COMPREHENSIVE HYPOTHESIS TESTING GUIDE (Python / SciPy / Statsmodels)
======================================================================
Covers: assumptions checks, parametric tests, non-parametric tests,
proportion tests, correlation tests, ANOVA, post-hoc tests, effect sizes,
power analysis, and multiple-comparison correction.
 
Install once:
    pip install scipy numpy pandas --break-system-packages
 
Pure SciPy/NumPy implementation — no statsmodels dependency required.
"""
 
import numpy as np
import pandas as pd
from scipy import stats
from itertools import combinations
 
np.random.seed(42)

In [6]:
# ======================================================================
# 0. THE FRAMEWORK — what hypothesis testing actually does
# ======================================================================
print(f"""
Every test follows the same logic:

1. State H0 (null: "no effect/no difference") and H1 (alternative).
2. Choose significance level alpha (usually 0.05).
3. Check the test's ASSUMPTIONS (normality, variance, independence...).
4. Compute a test statistic and its p-value.
5. p-value = P(seeing data this extreme | H0 is true).
   - p < alpha  -> reject H0 ("statistically significant")
   - p >= alpha -> fail to reject H0 (NOT "H0 is true")
6. Report an EFFECT SIZE alongside p-value — significance != importance.

Picking the right test depends on:

- Data type (continuous, ordinal, categorical/binary)
- Number of groups (1, 2, 3+)
- Independent or paired/repeated samples
- Whether normality/equal-variance assumptions hold
""")


Every test follows the same logic:

1. State H0 (null: "no effect/no difference") and H1 (alternative).
2. Choose significance level alpha (usually 0.05).
3. Check the test's ASSUMPTIONS (normality, variance, independence...).
4. Compute a test statistic and its p-value.
5. p-value = P(seeing data this extreme | H0 is true).
   - p < alpha  -> reject H0 ("statistically significant")
   - p >= alpha -> fail to reject H0 (NOT "H0 is true")
6. Report an EFFECT SIZE alongside p-value — significance != importance.

Picking the right test depends on:

- Data type (continuous, ordinal, categorical/binary)
- Number of groups (1, 2, 3+)
- Independent or paired/repeated samples
- Whether normality/equal-variance assumptions hold



In [7]:

# ======================================================================
# 1. ASSUMPTION CHECKS (run these BEFORE choosing a parametric test)
# ======================================================================
 
def check_normality(data, name="sample"):
    """Shapiro-Wilk: H0 = data is normally distributed.
    Best for n < 50; for larger n use D'Agostino-Pearson."""
    stat, p = stats.shapiro(data)
    print(f"[Shapiro-Wilk] {name}: stat={stat:.4f}, p={p:.4f} -> "
          f"{'normal' if p > 0.05 else 'NOT normal'} (alpha=0.05)")
    return p > 0.05
 
def check_equal_variance(a, b, name="groups"):
    """Levene's test: H0 = variances are equal. More robust to
    non-normality than Bartlett's test."""
    stat, p = stats.levene(a, b)
    print(f"[Levene] {name}: stat={stat:.4f}, p={p:.4f} -> "
          f"{'equal variance' if p > 0.05 else 'UNEQUAL variance'}")
    return p > 0.05
 
# Example data
group_a = np.random.normal(50, 10, 30)
group_b = np.random.normal(55, 12, 30)
 
check_normality(group_a, "group_a")
check_normality(group_b, "group_b")
check_equal_variance(group_a, group_b)

[Shapiro-Wilk] group_a: stat=0.9751, p=0.6868 -> normal (alpha=0.05)
[Shapiro-Wilk] group_b: stat=0.9837, p=0.9130 -> normal (alpha=0.05)
[Levene] groups: stat=2.0687, p=0.1557 -> equal variance


np.True_

In [8]:
# ======================================================================
# 2. ONE-SAMPLE TESTS — compare a sample to a known/hypothesized value
# ======================================================================
 
def one_sample_t_test(data, popmean, alpha=0.05):
    """Use when: continuous data, sample roughly normal, population
    std unknown (almost always the case)."""
    t_stat, p = stats.ttest_1samp(data, popmean)
    d = (np.mean(data) - popmean) / np.std(data, ddof=1)  # Cohen's d
    print(f"One-sample t-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f} "
          f"-> {'reject H0' if p < alpha else 'fail to reject H0'}")
    return t_stat, p
 
one_sample_t_test(group_a, popmean=48)
 
def one_sample_z_test(data, popmean, pop_std):
    """Use when population std IS known (rare) or n is large (>30)
    and you want to use the normal distribution instead of t."""
    n = len(data)
    z = (np.mean(data) - popmean) / (pop_std / np.sqrt(n))
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"One-sample z-test: z={z:.4f}, p={p:.4f}")
    return z, p

One-sample t-test: t=0.0721, p=0.9430, Cohen's d=0.013 -> fail to reject H0


In [9]:

# ======================================================================
# 3. TWO-SAMPLE TESTS — compare two groups
# ======================================================================
 
def independent_t_test(a, b, equal_var=True, alpha=0.05):
    """Independent (unpaired) samples, e.g. control vs treatment group,
    two different customer segments. equal_var=False -> Welch's t-test
    (safer default when variances differ or sample sizes differ)."""
    t_stat, p = stats.ttest_ind(a, b, equal_var=equal_var)
    pooled_std = np.sqrt(((len(a)-1)*np.var(a, ddof=1) +
                           (len(b)-1)*np.var(b, ddof=1)) / (len(a)+len(b)-2))
    d = (np.mean(a) - np.mean(b)) / pooled_std
    label = "Student's t" if equal_var else "Welch's t"
    print(f"{label}-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f} "
          f"-> {'reject H0' if p < alpha else 'fail to reject H0'}")
    return t_stat, p
 
independent_t_test(group_a, group_b, equal_var=True)   # standard
independent_t_test(group_a, group_b, equal_var=False)  # Welch's (recommended default)
 
def paired_t_test(before, after, alpha=0.05):
    """Same subjects measured twice, e.g. before/after an intervention,
    churn score pre/post a retention campaign."""
    t_stat, p = stats.ttest_rel(before, after)
    diff = np.array(after) - np.array(before)
    d = np.mean(diff) / np.std(diff, ddof=1)
    print(f"Paired t-test: t={t_stat:.4f}, p={p:.4f}, Cohen's d={d:.3f}")
    return t_stat, p
 
before = np.random.normal(60, 8, 25)
after = before + np.random.normal(3, 5, 25)  # simulate improvement
paired_t_test(before, after)

Student's t-test: t=-2.0720, p=0.0427, Cohen's d=-0.535 -> reject H0
Welch's t-test: t=-2.0720, p=0.0429, Cohen's d=-0.535 -> reject H0
Paired t-test: t=-3.8041, p=0.0009, Cohen's d=0.761


(np.float64(-3.804068700058591), np.float64(0.0008631807921069468))

In [10]:
# ======================================================================
# 4. NON-PARAMETRIC ALTERNATIVES (use when normality fails, or ordinal
#    data, or small/skewed samples)
# ======================================================================
 
def mann_whitney_u(a, b, alpha=0.05):
    """Non-parametric alternative to independent t-test. Compares
    distributions/medians via rank sums, no normality assumption."""
    stat, p = stats.mannwhitneyu(a, b, alternative='two-sided')
    print(f"Mann-Whitney U: U={stat:.4f}, p={p:.4f}")
    return stat, p
 
def wilcoxon_signed_rank(before, after, alpha=0.05):
    """Non-parametric alternative to paired t-test."""
    stat, p = stats.wilcoxon(before, after)
    print(f"Wilcoxon signed-rank: W={stat:.4f}, p={p:.4f}")
    return stat, p
 
def kruskal_wallis(*groups, alpha=0.05):
    """Non-parametric alternative to one-way ANOVA (3+ groups)."""
    stat, p = stats.kruskal(*groups)
    print(f"Kruskal-Wallis H: H={stat:.4f}, p={p:.4f}")
    return stat, p
 
mann_whitney_u(group_a, group_b)
wilcoxon_signed_rank(before, after)

Mann-Whitney U: U=310.0000, p=0.0392
Wilcoxon signed-rank: W=41.0000, p=0.0006


(np.float64(41.0), np.float64(0.0005564093589782715))

In [11]:

# ======================================================================
# 5. CATEGORICAL DATA — chi-square and proportion tests
# ======================================================================
 
def chi_square_independence(contingency_table, alpha=0.05):
    """H0: the two categorical variables are independent.
    e.g. does 'plan type' relate to 'churned (yes/no)'?"""
    chi2, p, dof, expected = stats.chi2_contingency(contingency_table)
    print(f"Chi-square test of independence: chi2={chi2:.4f}, dof={dof}, p={p:.4f}")
    print("Expected frequencies:\n", expected)
    if (expected < 5).any():
        print("WARNING: some expected counts < 5 — consider Fisher's exact test")
    return chi2, p
 
# Example: churn (rows) vs contract type (cols)
table = pd.DataFrame(
    [[120, 40, 10], [30, 60, 90]],
    index=["Churned", "Retained"],
    columns=["Month-to-month", "One-year", "Two-year"]
)
chi_square_independence(table.values)
 
def chi_square_goodness_of_fit(observed, expected, alpha=0.05):
    """H0: observed frequencies match an expected distribution."""
    chi2, p = stats.chisquare(observed, f_exp=expected)
    print(f"Chi-square GOF: chi2={chi2:.4f}, p={p:.4f}")
    return chi2, p
 
def fisher_exact_test(table_2x2, alpha=0.05):
    """Use for 2x2 tables with small expected counts (<5)."""
    odds_ratio, p = stats.fisher_exact(table_2x2)
    print(f"Fisher's exact: OR={odds_ratio:.4f}, p={p:.4f}")
    return odds_ratio, p
 
def two_proportion_z_test(count1, nobs1, count2, nobs2, alpha=0.05):
    """Compare two proportions/rates, e.g. conversion rate A vs B.
    Pooled-proportion z-test (standard A/B test formula)."""
    p1, p2 = count1 / nobs1, count2 / nobs2
    p_pool = (count1 + count2) / (nobs1 + nobs2)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/nobs1 + 1/nobs2))
    z = (p1 - p2) / se
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    print(f"Two-proportion z-test: p1={p1:.3f}, p2={p2:.3f}, z={z:.4f}, p={p:.4f}")
    return z, p
 
two_proportion_z_test(count1=45, nobs1=500, count2=60, nobs2=520)  # e.g. A/B test conversions
 
def mcnemar_test(table_2x2, alpha=0.05):
    """Paired categorical data, e.g. same customers' yes/no before vs
    after a UI change (McNemar, not chi-square, because paired).
    table_2x2 = [[both_yes, yes_to_no], [no_to_yes, both_no]]
    Only the off-diagonal discordant pairs (b, c) matter."""
    b, c = table_2x2[0][1], table_2x2[1][0]
    if b + c < 25:
        p = min(2 * stats.binom.cdf(min(b, c), b + c, 0.5), 1.0)  # exact binomial
        print(f"McNemar's test (exact): b={b}, c={c}, p={p:.4f}")
        return None, p
    stat = (abs(b - c) - 1)**2 / (b + c)  # chi2 approx with continuity correction
    p = 1 - stats.chi2.cdf(stat, df=1)
    print(f"McNemar's test (chi2 approx): stat={stat:.4f}, p={p:.4f}")
    return stat, p

Chi-square test of independence: chi2=121.8137, dof=2, p=0.0000
Expected frequencies:
 [[72.85714286 48.57142857 48.57142857]
 [77.14285714 51.42857143 51.42857143]]
Two-proportion z-test: p1=0.090, p2=0.115, z=-1.3337, p=0.1823


In [12]:

# ======================================================================
# 6. ANOVA — comparing 3+ group means
# ======================================================================
 
def one_way_anova(*groups, alpha=0.05):
    """H0: all group means are equal. Assumes normality + equal
    variance across groups (check with Levene's test first)."""
    f_stat, p = stats.f_oneway(*groups)
    # eta-squared effect size
    all_data = np.concatenate(groups)
    grand_mean = np.mean(all_data)
    ss_between = sum(len(g) * (np.mean(g) - grand_mean)**2 for g in groups)
    ss_total = sum((all_data - grand_mean)**2)
    eta_sq = ss_between / ss_total
    print(f"One-way ANOVA: F={f_stat:.4f}, p={p:.4f}, eta^2={eta_sq:.3f}")
    return f_stat, p
 
group_c = np.random.normal(52, 9, 30)
one_way_anova(group_a, group_b, group_c)
 
def posthoc_pairwise(groups_dict, alpha=0.05):
    """Run AFTER a significant ANOVA to find WHICH pairs differ.
    Pairwise independent t-tests with Bonferroni correction
    (a practical stand-in for Tukey HSD; Tukey is slightly less
    conservative but needs statsmodels)."""
    names = list(groups_dict.keys())
    pairs = list(combinations(names, 2))
    raw_p = []
    for n1, n2 in pairs:
        _, p = stats.ttest_ind(groups_dict[n1], groups_dict[n2])
        raw_p.append(p)
    corrected = np.minimum(np.array(raw_p) * len(pairs), 1.0)  # Bonferroni
    for (n1, n2), p_raw, p_corr in zip(pairs, raw_p, corrected):
        flag = "significant" if p_corr < alpha else "n.s."
        print(f"{n1} vs {n2}: raw p={p_raw:.4f}, Bonferroni p={p_corr:.4f} -> {flag}")
    return dict(zip(pairs, corrected))
 
posthoc_pairwise({"A": group_a, "B": group_b, "C": group_c})
 
def two_way_anova_note():
    """Two-way ANOVA (two categorical factors + interaction) needs a
    full linear-model decomposition that's impractical to hand-roll
    reliably. If you need it: pip install statsmodels, then:
 
        from statsmodels.formula.api import ols
        import statsmodels.api as sm
        model = ols('dv ~ C(factor1) * C(factor2)', data=df).fit()
        print(sm.stats.anova_lm(model, typ=2))
    """
    print("Two-way ANOVA requires statsmodels — see function docstring.")
 
two_way_anova_note()

One-way ANOVA: F=2.2617, p=0.1103, eta^2=0.049
A vs B: raw p=0.0427, Bonferroni p=0.1282 -> n.s.
A vs C: raw p=0.1442, Bonferroni p=0.4326 -> n.s.
B vs C: raw p=0.5089, Bonferroni p=1.0000 -> n.s.
Two-way ANOVA requires statsmodels — see function docstring.


In [13]:

# ======================================================================
# 7. CORRELATION TESTS
# ======================================================================
 
def pearson_correlation(x, y, alpha=0.05):
    """Linear relationship, both variables continuous & normal."""
    r, p = stats.pearsonr(x, y)
    print(f"Pearson r={r:.4f}, p={p:.4f}")
    return r, p
 
def spearman_correlation(x, y, alpha=0.05):
    """Monotonic relationship, ordinal data or non-normal, robust to
    outliers (rank-based)."""
    rho, p = stats.spearmanr(x, y)
    print(f"Spearman rho={rho:.4f}, p={p:.4f}")
    return rho, p
 
x = np.random.normal(0, 1, 50)
y = 2*x + np.random.normal(0, 1, 50)
pearson_correlation(x, y)
spearman_correlation(x, y)

Pearson r=0.8698, p=0.0000
Spearman rho=0.8648, p=0.0000


(np.float64(0.8647779111644658), np.float64(5.747304523217858e-16))

In [14]:
# ======================================================================
# 8. MULTIPLE COMPARISONS CORRECTION
# ======================================================================
"""
Running many tests inflates false-positive risk (e.g. 20 tests at
alpha=0.05 gives ~64% chance of at least one false positive).
Correct with Bonferroni (strict) or Benjamini-Hochberg/FDR (less strict,
better for exploratory analysis with many tests).
"""
 
def bonferroni_correction(pvalues, alpha=0.05):
    """Strictest correction: multiply each p-value by the number of tests."""
    pvalues = np.array(pvalues)
    corrected = np.minimum(pvalues * len(pvalues), 1.0)
    reject = corrected < alpha
    print(f"Bonferroni corrected p-values: {corrected}")
    print(f"Reject H0: {reject}")
    return corrected, reject
 
def benjamini_hochberg_correction(pvalues, alpha=0.05):
    """FDR control — less conservative than Bonferroni, standard choice
    for exploratory analysis with many simultaneous tests."""
    pvalues = np.array(pvalues)
    n = len(pvalues)
    order = np.argsort(pvalues)
    ranked = pvalues[order]
    bh_critical = (np.arange(1, n + 1) / n) * alpha
    below = ranked <= bh_critical
    corrected = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    corrected = np.minimum(corrected, 1.0)
    # restore original order
    out_corrected = np.empty(n)
    out_corrected[order] = corrected
    reject = out_corrected < alpha
    print(f"Benjamini-Hochberg corrected p-values: {out_corrected}")
    print(f"Reject H0: {reject}")
    return out_corrected, reject
 
bonferroni_correction([0.01, 0.04, 0.03, 0.20, 0.001])
benjamini_hochberg_correction([0.01, 0.04, 0.03, 0.20, 0.001])

Bonferroni corrected p-values: [0.05  0.2   0.15  1.    0.005]
Reject H0: [False False False False  True]
Benjamini-Hochberg corrected p-values: [0.025 0.05  0.05  0.2   0.005]
Reject H0: [ True False  True False  True]


(array([0.025, 0.05 , 0.05 , 0.2  , 0.005]),
 array([ True, False,  True, False,  True]))

In [15]:
# ======================================================================
# 9. POWER ANALYSIS — how big a sample do you need?
# ======================================================================
"""
Statistical power = P(correctly rejecting H0 when H1 is true).
Standard target: power = 0.80. Run this BEFORE collecting data.
"""
 
def sample_size_for_t_test(effect_size, alpha=0.05, target_power=0.8):
    """Analytic normal-approximation formula for a two-sample t-test
    (Cohen's formula). Close enough to the exact noncentral-t solution
    for planning purposes; use statsmodels' TTestIndPower for the
    exact value if needed."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    z_beta = stats.norm.ppf(target_power)
    n = 2 * ((z_alpha + z_beta) / effect_size) ** 2
    print(f"Required sample size per group: {np.ceil(n):.0f} "
          f"(effect size d={effect_size}, power={target_power})")
    return n
 
sample_size_for_t_test(effect_size=0.5)  # medium effect (Cohen's convention)
 
def achieved_power(n_per_group, effect_size, alpha=0.05):
    """Approximate achieved power for a two-sample t-test given n."""
    z_alpha = stats.norm.ppf(1 - alpha / 2)
    ncp = effect_size * np.sqrt(n_per_group / 2)
    pw = 1 - stats.norm.cdf(z_alpha - ncp) + stats.norm.cdf(-z_alpha - ncp)
    print(f"Achieved power with n={n_per_group}, d={effect_size}: {pw:.3f}")
    return pw
 
achieved_power(30, 0.5)

Required sample size per group: 63 (effect size d=0.5, power=0.8)
Achieved power with n=30, d=0.5: 0.491


np.float64(0.4906855676699691)

In [16]:
# ======================================================================
# 10. DECISION CHEAT SHEET
# ======================================================================
"""
QUESTION                              -> TEST
---------------------------------------------------------------------
1 sample mean vs known value          -> one-sample t-test (or z-test if
                                          pop std known)
2 independent group means             -> Welch's t-test (default) /
                                          Mann-Whitney U if non-normal
2 paired/matched means                -> paired t-test / Wilcoxon
                                          signed-rank if non-normal
3+ independent group means            -> one-way ANOVA + pairwise post-hoc /
                                          Kruskal-Wallis if non-normal
2 factors, group means                -> two-way ANOVA (needs statsmodels)
2 categorical variables, independence -> chi-square / Fisher's exact
                                          (small expected counts)
2 paired categorical (binary)         -> McNemar's test
2 proportions/rates                   -> two-proportion z-test
Linear association, continuous        -> Pearson correlation
Monotonic/ordinal association         -> Spearman correlation
Many p-values at once                 -> Bonferroni / Benjamini-Hochberg
Planning sample size                  -> power analysis (Cohen's formula)
 
ALWAYS: check assumptions (normality via Shapiro-Wilk, equal variance
via Levene) → pick parametric or non-parametric version → report
effect size, not just p-value → correct for multiple comparisons if
running many tests.
"""
 
print("\nAll examples executed. See section 10 for the decision cheat sheet.")


All examples executed. See section 10 for the decision cheat sheet.


In [18]:
print("""
HYPOTHESIS TESTING GUIDE — PART 2 (advanced / previously-missing topics)
=========================================================================
Everything here is built from raw numpy/scipy math so you can see what's
actually happening under the hood — this is exactly what statsmodels
does internally, just spelled out. Good for VS Code, step through with
a debugger and watch the arrays.

Covers: two-way ANOVA, repeated-measures ANOVA, regression-based tests,
bootstrap tests, permutation tests, Bayesian hypothesis testing,
time-series tests (stationarity, autocorrelation), survival analysis
(Kaplan-Meier, log-rank test).

Run part 1 first (hypothesis_testing_guide.py) for the core test suite
and decision cheat sheet.
""")


HYPOTHESIS TESTING GUIDE — PART 2 (advanced / previously-missing topics)
Everything here is built from raw numpy/scipy math so you can see what's
actually happening under the hood — this is exactly what statsmodels
does internally, just spelled out. Good for VS Code, step through with
a debugger and watch the arrays.

Covers: two-way ANOVA, repeated-measures ANOVA, regression-based tests,
bootstrap tests, permutation tests, Bayesian hypothesis testing,
time-series tests (stationarity, autocorrelation), survival analysis
(Kaplan-Meier, log-rank test).

Run part 1 first (hypothesis_testing_guide.py) for the core test suite
and decision cheat sheet.



In [19]:

 
# ======================================================================
# 11. TWO-WAY ANOVA — two categorical factors + their interaction
# ======================================================================
"""
Question shape: does churn differ by CONTRACT TYPE, by REGION, and does
the *combination* of the two matter (interaction effect)?
 
We manually decompose total variance into: factor A, factor B, A*B
interaction, and residual (error) — this is what 'Type II/III sums of
squares' means in statsmodels output.
"""
 
def two_way_anova(df, dv, factor1, factor2, alpha=0.05):
    """df must have columns [dv, factor1, factor2], balanced or not.
    Returns a table of F-stats and p-values for each term."""
    grand_mean = df[dv].mean()
    levels1 = df[factor1].unique()
    levels2 = df[factor2].unique()
 
    # Sum of squares total
    ss_total = ((df[dv] - grand_mean) ** 2).sum()
 
    # SS for factor 1 (main effect)
    ss_a = sum(
        len(df[df[factor1] == l1]) * (df[df[factor1] == l1][dv].mean() - grand_mean) ** 2
        for l1 in levels1
    )
    # SS for factor 2 (main effect)
    ss_b = sum(
        len(df[df[factor2] == l2]) * (df[df[factor2] == l2][dv].mean() - grand_mean) ** 2
        for l2 in levels2
    )
    # SS for interaction: cell means vs. what main effects alone predict
    ss_ab = 0
    for l1 in levels1:
        for l2 in levels2:
            cell = df[(df[factor1] == l1) & (df[factor2] == l2)]
            if len(cell) == 0:
                continue
            cell_mean = cell[dv].mean()
            mean1 = df[df[factor1] == l1][dv].mean()
            mean2 = df[df[factor2] == l2][dv].mean()
            predicted = mean1 + mean2 - grand_mean
            ss_ab += len(cell) * (cell_mean - predicted) ** 2
 
    ss_error = ss_total - ss_a - ss_b - ss_ab
 
    df_a, df_b = len(levels1) - 1, len(levels2) - 1
    df_ab = df_a * df_b
    df_error = len(df) - len(levels1) * len(levels2)
 
    ms_a, ms_b = ss_a / df_a, ss_b / df_b
    ms_ab = ss_ab / df_ab
    ms_error = ss_error / df_error
 
    f_a, f_b, f_ab = ms_a / ms_error, ms_b / ms_error, ms_ab / ms_error
    p_a = 1 - stats.f.cdf(f_a, df_a, df_error)
    p_b = 1 - stats.f.cdf(f_b, df_b, df_error)
    p_ab = 1 - stats.f.cdf(f_ab, df_ab, df_error)
 
    result = pd.DataFrame({
        "source": [factor1, factor2, f"{factor1} x {factor2}", "residual"],
        "SS": [ss_a, ss_b, ss_ab, ss_error],
        "df": [df_a, df_b, df_ab, df_error],
        "F": [f_a, f_b, f_ab, np.nan],
        "p": [p_a, p_b, p_ab, np.nan],
    })
    print(result.to_string(index=False))
    return result
 
# Example: monthly usage by contract type and region
np.random.seed(1)
n_per_cell = 20
rows = []
for contract in ["Monthly", "Annual"]:
    for region in ["Urban", "Rural"]:
        base = 50 + (10 if contract == "Annual" else 0) + (5 if region == "Urban" else 0)
        interaction_bump = 8 if (contract == "Annual" and region == "Urban") else 0
        vals = np.random.normal(base + interaction_bump, 8, n_per_cell)
        for v in vals:
            rows.append({"usage": v, "contract": contract, "region": region})
usage_df = pd.DataFrame(rows)
two_way_anova(usage_df, "usage", "contract", "region")

           source          SS  df         F            p
         contract 5265.455590   1 87.499553 2.786660e-14
           region 1539.689393   1 25.586036 2.854394e-06
contract x region  400.395444   1  6.653636 1.182577e-02
         residual 4573.447626  76       NaN          NaN


,source,SS,df,F,p
0,contract,5265.455590,1,87.499553,2.786660e-14
1,region,1539.689393,1,25.586036,2.854394e-06
2,contract x region,400.395444,1,6.653636,1.182577e-02
3,residual,4573.447626,76,NaN,NaN


In [20]:

# ======================================================================
# 12. REPEATED-MEASURES ANOVA — 3+ measurements on the SAME subjects
# ======================================================================
"""
Use when: one group measured at 3+ time points (e.g. churn-risk score
at month 1, 3, 6 for the same customers) — NOT independent groups.
Ignoring the repeated-measures structure and running a normal one-way
ANOVA overstates your degrees of freedom and inflates false positives.
"""
 
def repeated_measures_anova(data_wide, alpha=0.05):
    """data_wide: 2D array, rows=subjects, columns=repeated conditions."""
    data_wide = np.asarray(data_wide)
    n_subjects, n_conditions = data_wide.shape
    grand_mean = data_wide.mean()
 
    ss_total = ((data_wide - grand_mean) ** 2).sum()
    ss_conditions = n_subjects * ((data_wide.mean(axis=0) - grand_mean) ** 2).sum()
    ss_subjects = n_conditions * ((data_wide.mean(axis=1) - grand_mean) ** 2).sum()
    ss_error = ss_total - ss_conditions - ss_subjects
 
    df_cond = n_conditions - 1
    df_error = (n_subjects - 1) * (n_conditions - 1)
 
    ms_cond = ss_conditions / df_cond
    ms_error = ss_error / df_error
    f_stat = ms_cond / ms_error
    p = 1 - stats.f.cdf(f_stat, df_cond, df_error)
 
    print(f"Repeated-measures ANOVA: F({df_cond},{df_error})={f_stat:.4f}, p={p:.4f}")
    return f_stat, p
 
# Example: churn-risk score for 25 customers at month 1, 3, 6
n_subj = 25
month1 = np.random.normal(40, 8, n_subj)
month3 = month1 + np.random.normal(-3, 4, n_subj)   # risk drops slightly
month6 = month3 + np.random.normal(-2, 4, n_subj)
repeated_measures_anova(np.column_stack([month1, month3, month6]))

Repeated-measures ANOVA: F(2,48)=13.3971, p=0.0000


(np.float64(13.397088939142698), np.float64(2.3821146947322624e-05))

In [21]:
# ======================================================================
# 13. REGRESSION-BASED HYPOTHESIS TESTS
# ======================================================================
"""
Every t-test/ANOVA above is secretly a special case of linear
regression. Testing 'does X predict Y' via regression coefficients is
more flexible: handles continuous + categorical predictors together,
controls for confounders.
 
H0 for a coefficient: beta_j = 0 (predictor has no linear effect on Y,
holding others constant).
"""
 
def ols_regression_test(X, y, feature_names=None, alpha=0.05):
    """Manual OLS with t-tests on coefficients (no statsmodels needed).
    X: (n, p) design matrix WITHOUT intercept column (added automatically)."""
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n, p = X.shape
    X_design = np.column_stack([np.ones(n), X])  # add intercept
    k = X_design.shape[1]
 
    # OLS estimate: beta = (X'X)^-1 X'y
    XtX_inv = np.linalg.inv(X_design.T @ X_design)
    beta = XtX_inv @ X_design.T @ y
 
    y_hat = X_design @ beta
    residuals = y - y_hat
    ss_res = (residuals ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    r_squared = 1 - ss_res / ss_tot
    df_resid = n - k
 
    sigma_sq = ss_res / df_resid
    se_beta = np.sqrt(np.diag(sigma_sq * XtX_inv))
    t_stats = beta / se_beta
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stats), df_resid))
 
    names = ["Intercept"] + (feature_names or [f"X{i+1}" for i in range(p)])
    result = pd.DataFrame({
        "coef": beta, "std_err": se_beta, "t": t_stats, "p_value": p_values
    }, index=names)
 
    # Overall model F-test: H0 = ALL coefficients (except intercept) are 0
    ss_model = ss_tot - ss_res
    df_model = k - 1
    f_stat = (ss_model / df_model) / (ss_res / df_resid)
    f_p = 1 - stats.f.cdf(f_stat, df_model, df_resid)
 
    print(result)
    print(f"\nR^2 = {r_squared:.4f}")
    print(f"Overall F-test: F({df_model},{df_resid})={f_stat:.4f}, p={f_p:.4f} "
          f"-> {'model explains significant variance' if f_p < alpha else 'not significant'}")
    return result, r_squared, (f_stat, f_p)
 
# Example: predict monthly spend from tenure (months) and support_tickets
n = 100
tenure = np.random.uniform(1, 60, n)
support_tickets = np.random.poisson(3, n)
spend = 20 + 1.5 * tenure - 3 * support_tickets + np.random.normal(0, 15, n)
ols_regression_test(np.column_stack([tenure, support_tickets]), spend,
                     feature_names=["tenure_months", "support_tickets"])
 
def nested_model_f_test(ss_res_reduced, ss_res_full, df_reduced_params,
                         df_full_params, n, alpha=0.05):
    """Compare a smaller (reduced) model to a larger (full) model that
    nests it — e.g. does adding 'region' meaningfully improve the model
    beyond tenure + support_tickets alone?
    H0: the extra predictors in the full model add nothing."""
    df1 = df_full_params - df_reduced_params        # extra params
    df2 = n - df_full_params
    f_stat = ((ss_res_reduced - ss_res_full) / df1) / (ss_res_full / df2)
    p = 1 - stats.f.cdf(f_stat, df1, df2)
    print(f"Nested model F-test: F({df1},{df2})={f_stat:.4f}, p={p:.4f}")
    return f_stat, p

                      coef   std_err          t       p_value
Intercept        23.655910  4.485095   5.274339  8.093216e-07
tenure_months     1.539547  0.089700  17.163287  0.000000e+00
support_tickets  -4.142140  0.859256  -4.820610  5.296476e-06

R^2 = 0.7864
Overall F-test: F(2,97)=178.5643, p=0.0000 -> model explains significant variance


In [22]:
# ======================================================================
# 14. BOOTSTRAP HYPOTHESIS TESTS
# ======================================================================
"""
Use when: you don't trust the normality/distributional assumptions of
a classical test, or you want a confidence interval for a statistic
that has no clean formula (e.g. median difference, ratio of variances).
Idea: resample your own data (with replacement) thousands of times to
build an empirical sampling distribution — no formula needed.
"""
 
def bootstrap_diff_in_means(a, b, n_boot=10000, alpha=0.05, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    observed_diff = a.mean() - b.mean()
 
    boot_diffs = np.empty(n_boot)
    for i in range(n_boot):
        boot_a = rng.choice(a, size=len(a), replace=True)
        boot_b = rng.choice(b, size=len(b), replace=True)
        boot_diffs[i] = boot_a.mean() - boot_b.mean()
 
    ci_low, ci_high = np.percentile(boot_diffs, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    # two-sided p-value: how often bootstrap distribution centered at 0 exceeds observed
    null_shifted = boot_diffs - boot_diffs.mean()
    p = np.mean(np.abs(null_shifted) >= np.abs(observed_diff))
 
    print(f"Bootstrap diff-in-means: observed={observed_diff:.4f}, "
          f"{100*(1-alpha):.0f}% CI=[{ci_low:.4f}, {ci_high:.4f}], p~={p:.4f}")
    return observed_diff, (ci_low, ci_high), p
 
bootstrap_diff_in_means(group_a := np.random.normal(50, 10, 30),
                         group_b := np.random.normal(55, 12, 30))

Bootstrap diff-in-means: observed=-6.1036, 95% CI=[-12.4302, -0.1518], p~=0.0508


(np.float64(-6.103600635146272),
 (np.float64(-12.430225215562789), np.float64(-0.1518042333787234)),
 np.float64(0.0508))

In [23]:
 
# ======================================================================
# 15. PERMUTATION TESTS (randomization tests)
# ======================================================================
"""
Use when: you want a test that makes almost NO distributional
assumptions at all. Idea: under H0, group labels are meaningless —
shuffle them thousands of times and see how extreme the real grouping
is compared to random groupings. Great for small/weird samples.
"""
 
def permutation_test_diff_in_means(a, b, n_perm=10000, seed=42):
    rng = np.random.default_rng(seed)
    a, b = np.asarray(a), np.asarray(b)
    observed_diff = a.mean() - b.mean()
    combined = np.concatenate([a, b])
    n_a = len(a)
 
    perm_diffs = np.empty(n_perm)
    for i in range(n_perm):
        shuffled = rng.permutation(combined)
        perm_diffs[i] = shuffled[:n_a].mean() - shuffled[n_a:].mean()
 
    p = np.mean(np.abs(perm_diffs) >= np.abs(observed_diff))
    print(f"Permutation test: observed diff={observed_diff:.4f}, p={p:.4f}")
    return observed_diff, p
 
permutation_test_diff_in_means(group_a, group_b)

Permutation test: observed diff=-6.1036, p=0.0623


(np.float64(-6.103600635146272), np.float64(0.0623))

In [24]:
# ======================================================================
# 16. BAYESIAN HYPOTHESIS TESTING (conceptual intro, no PyMC required)
# ======================================================================
"""
Different philosophy from everything above. Instead of a single
p-value, you get a full probability distribution over the parameter,
updated from a prior belief using observed data (Bayes' theorem).
 
Key outputs: posterior distribution, credible interval (NOT the same
as a confidence interval — a 95% credible interval means there's a
95% probability the true value is in that range, which is what people
often WRONGLY think a p-value/CI tells them).
"""
 
def bayesian_proportion_test(successes, n, prior_alpha=1, prior_beta=1):
    """Beta-Binomial conjugate model for a proportion, e.g. conversion
    rate. prior_alpha=prior_beta=1 is a flat/uninformative prior."""
    post_alpha = prior_alpha + successes
    post_beta = prior_beta + (n - successes)
    posterior = stats.beta(post_alpha, post_beta)
 
    mean = posterior.mean()
    ci_low, ci_high = posterior.ppf([0.025, 0.975])
    print(f"Bayesian proportion estimate: mean={mean:.4f}, "
          f"95% credible interval=[{ci_low:.4f}, {ci_high:.4f}]")
    return posterior
 
def bayesian_compare_two_proportions(successes_a, n_a, successes_b, n_b, n_samples=100000, seed=42):
    """P(rate_A > rate_B) via Monte Carlo sampling from both posteriors —
    directly answers 'how likely is A actually better than B', which a
    p-value never tells you."""
    rng = np.random.default_rng(seed)
    post_a = stats.beta(1 + successes_a, 1 + n_a - successes_a)
    post_b = stats.beta(1 + successes_b, 1 + n_b - successes_b)
    samples_a = post_a.rvs(n_samples, random_state=rng)
    samples_b = post_b.rvs(n_samples, random_state=rng)
    prob_a_better = np.mean(samples_a > samples_b)
    print(f"P(rate_A > rate_B) = {prob_a_better:.4f}")
    return prob_a_better
 
bayesian_proportion_test(successes=45, n=500)
bayesian_compare_two_proportions(45, 500, 60, 520)
 
def bayes_factor_bic_approx(ss_res_null, ss_res_alt, n, k_null, k_alt):
    """Rough Bayes Factor approximation using BIC difference — a
    practical shortcut when a full Bayesian model is overkill.
    BF > 1 favors the alternative model; BF > 3 = 'moderate' evidence,
    BF > 10 = 'strong' evidence (Kass & Raftery guideline)."""
    bic_null = n * np.log(ss_res_null / n) + k_null * np.log(n)
    bic_alt = n * np.log(ss_res_alt / n) + k_alt * np.log(n)
    bf_10 = np.exp((bic_null - bic_alt) / 2)
    print(f"Approx Bayes Factor (alt vs null) = {bf_10:.3f}")
    return bf_10

Bayesian proportion estimate: mean=0.0916, 95% credible interval=[0.0680, 0.1183]
P(rate_A > rate_B) = 0.0920


In [25]:
# ======================================================================
# 17. TIME-SERIES TESTS
# ======================================================================
"""
Use when: your data is a sequence over time (e.g. monthly churn rate,
daily active users) — standard tests assume independent observations,
which time series violate by design (today correlates with yesterday).
"""
 
def augmented_dickey_fuller_manual(series, lags=1):
    """Tests H0: series has a unit root (i.e. is NON-stationary —
    mean/variance drift over time, common in raw time series).
    Stationarity matters before trusting many time-series models.
    This is a simplified manual version of the ADF test regression;
    for production use `from statsmodels.tsa.stattools import adfuller`."""
    series = np.asarray(series, dtype=float)
    y = np.diff(series)
    y_lag1 = series[:-1]
    n = len(y)
 
    X_parts = [y_lag1]
    for l in range(1, lags + 1):
        if l < len(y):
            lagged_diff = np.concatenate([np.full(l, np.nan), np.diff(series)[:-l]])
            X_parts.append(lagged_diff[:n])
    X = np.column_stack(X_parts)
    valid = ~np.isnan(X).any(axis=1)
    X, y_valid = X[valid], y[valid]
    X_design = np.column_stack([np.ones(len(y_valid)), X])
 
    beta = np.linalg.lstsq(X_design, y_valid, rcond=None)[0]
    residuals = y_valid - X_design @ beta
    df_resid = len(y_valid) - X_design.shape[1]
    sigma_sq = (residuals ** 2).sum() / df_resid
    se = np.sqrt(sigma_sq * np.linalg.inv(X_design.T @ X_design)[1, 1])
    adf_stat = beta[1] / se
 
    # ADF has non-standard critical values (not a normal t-distribution);
    # rough 5% critical value for no-trend case is about -2.86
    print(f"ADF statistic: {adf_stat:.4f} (rough 5% critical value ~ -2.86)")
    print("-> " + ("likely stationary" if adf_stat < -2.86 else "likely NON-stationary (has unit root)"))
    return adf_stat
 
stationary_series = np.random.normal(0, 1, 100)
trending_series = np.cumsum(np.random.normal(0.3, 1, 100))  # random walk with drift
augmented_dickey_fuller_manual(stationary_series)
augmented_dickey_fuller_manual(trending_series)
 
def ljung_box_test(residuals, lags=10, alpha=0.05):
    """Tests H0: no autocorrelation in residuals up to given lag.
    Use AFTER fitting a model (e.g. regression, ARIMA) to check the
    residuals are 'white noise' — if not, the model is missing
    structure in the data."""
    residuals = np.asarray(residuals)
    n = len(residuals)
    acf = [np.corrcoef(residuals[:-k], residuals[k:])[0, 1] for k in range(1, lags + 1)]
    q_stat = n * (n + 2) * sum((r ** 2) / (n - k) for k, r in enumerate(acf, start=1))
    p = 1 - stats.chi2.cdf(q_stat, lags)
    print(f"Ljung-Box Q({lags})={q_stat:.4f}, p={p:.4f} -> "
          f"{'autocorrelation present' if p < alpha else 'no significant autocorrelation'}")
    return q_stat, p
 
ljung_box_test(np.random.normal(0, 1, 100))  # white noise -> should NOT reject

ADF statistic: -7.2532 (rough 5% critical value ~ -2.86)
-> likely stationary
ADF statistic: -0.2932 (rough 5% critical value ~ -2.86)
-> likely NON-stationary (has unit root)
Ljung-Box Q(10)=19.2236, p=0.0375 -> autocorrelation present


(np.float64(19.223605766397093), np.float64(0.037512764667671394))

In [26]:
# ======================================================================
# 18. SURVIVAL ANALYSIS — Kaplan-Meier + log-rank test
# ======================================================================
"""
This is the RIGHT framework for churn — not just 'did they churn
yes/no' but 'HOW LONG until they churned', while correctly handling
customers who haven't churned yet (censored data). Ignoring censoring
(e.g. dropping still-active customers) biases every other test above.
"""
 
def kaplan_meier(durations, event_observed):
    """durations: time to event or censoring.
    event_observed: 1 if churn observed, 0 if censored (still active).
    Returns survival probability at each observed event time."""
    durations = np.asarray(durations)
    event_observed = np.asarray(event_observed)
    order = np.argsort(durations)
    durations, event_observed = durations[order], event_observed[order]
 
    unique_times = np.unique(durations[event_observed == 1])
    n_at_risk = len(durations)
    survival = 1.0
    km_table = []
    for t in unique_times:
        at_risk = np.sum(durations >= t)
        events = np.sum((durations == t) & (event_observed == 1))
        survival *= (1 - events / at_risk)
        km_table.append((t, at_risk, events, survival))
 
    km_df = pd.DataFrame(km_table, columns=["time", "at_risk", "events", "survival_prob"])
    print(km_df.to_string(index=False))
    return km_df
 
def log_rank_test(durations_a, events_a, durations_b, events_b, alpha=0.05):
    """H0: two groups have identical survival curves, e.g. does a
    retention-campaign group churn later than a control group?
    (This is the survival-analysis equivalent of a two-sample test.)"""
    data = pd.DataFrame({
        "time": np.concatenate([durations_a, durations_b]),
        "event": np.concatenate([events_a, events_b]),
        "group": ["A"] * len(durations_a) + ["B"] * len(durations_b),
    })
    event_times = np.sort(data.loc[data.event == 1, "time"].unique())
 
    O_a, E_a, V = 0.0, 0.0, 0.0
    for t in event_times:
        at_risk = data[data.time >= t]
        at_risk_a = at_risk[at_risk.group == "A"]
        n = len(at_risk)
        n_a = len(at_risk_a)
        d = ((data.time == t) & (data.event == 1)).sum()
        d_a = ((data.time == t) & (data.event == 1) & (data.group == "A")).sum()
        if n <= 1:
            continue
        O_a += d_a
        E_a += d * n_a / n
        V += d * (n_a / n) * (1 - n_a / n) * (n - d) / (n - 1) if n > 1 else 0
 
    chi2_stat = (O_a - E_a) ** 2 / V if V > 0 else 0
    p = 1 - stats.chi2.cdf(chi2_stat, df=1)
    print(f"Log-rank test: chi2={chi2_stat:.4f}, p={p:.4f} -> "
          f"{'survival curves differ significantly' if p < alpha else 'no significant difference'}")
    return chi2_stat, p
 
# Example: retention campaign (A) vs control (B), months-to-churn
rng = np.random.default_rng(7)
durations_control = rng.exponential(12, 60)       # avg 12 months to churn
events_control = (durations_control < 24).astype(int)  # censored after 24mo observation
durations_control = np.minimum(durations_control, 24)
 
durations_treated = rng.exponential(18, 60)       # campaign delays churn -> avg 18mo
events_treated = (durations_treated < 24).astype(int)
durations_treated = np.minimum(durations_treated, 24)
 
print("\n-- Kaplan-Meier, control group --")
kaplan_meier(durations_control, events_control)
print("\n-- Log-rank test: treated vs control --")
log_rank_test(durations_treated, events_treated, durations_control, events_control)


-- Kaplan-Meier, control group --
     time  at_risk  events  survival_prob
 0.099608       60       1       0.983333
 0.117044       59       1       0.966667
 0.266030       58       1       0.950000
 0.720556       57       1       0.933333
 0.824408       56       1       0.916667
 0.900738       55       1       0.900000
 1.178416       54       1       0.883333
 1.526601       53       1       0.866667
 1.739358       52       1       0.850000
 2.478393       51       1       0.833333
 2.664855       50       1       0.816667
 2.865293       49       1       0.800000
 3.004144       48       1       0.783333
 3.519345       47       1       0.766667
 3.606408       46       1       0.750000
 3.630646       45       1       0.733333
 3.745747       44       1       0.716667
 4.180475       43       1       0.700000
 4.267534       42       1       0.683333
 4.606255       41       1       0.666667
 5.701847       40       1       0.650000
 6.255818       39       1       0.633333

(np.float64(5.846422873932343), np.float64(0.015608747609590967))

In [27]:
# ======================================================================
# 19. EXTENDED DECISION CHEAT SHEET
# ======================================================================
"""
QUESTION                                  -> TEST
-----------------------------------------------------------------------
2 factors affecting a continuous outcome  -> two-way ANOVA (+ interaction)
3+ repeated measurements, same subjects   -> repeated-measures ANOVA
Does X (continuous/categorical) predict Y -> OLS regression + coefficient
                                              t-tests
Does adding predictors improve the model  -> nested model F-test
Assumptions violated / no clean formula   -> bootstrap (CI or diff test)
Want minimal distributional assumptions   -> permutation test
Want P(hypothesis true), not just p-value -> Bayesian test (posterior,
                                              credible interval, Bayes factor)
Is a time series stationary               -> Augmented Dickey-Fuller
Are model residuals autocorrelated        -> Ljung-Box test
Time-to-event with censored data (churn!) -> Kaplan-Meier + log-rank test
 
RULE OF THUMB FOR YOUR CHURN PLATFORM:
- "Did feature X change usage?" (continuous)         -> t-test / regression
- "Did feature X change churn RATE?" (binary)         -> two-proportion z /
                                                          chi-square
- "Did feature X change TIME-TO-churn?" (the real Q)  -> Kaplan-Meier +
                                                          log-rank test
Most churn platforms answer the wrong question (rate)
when they actually care about the third one (timing).
"""
 
print("\nPart 2 complete. Combine with Part 1 for the full toolkit.")


Part 2 complete. Combine with Part 1 for the full toolkit.
